In [2]:
import re
from collections import defaultdict
from typing import List
from utils.exploit_gates2 import NetlistParser

In [3]:
Test_Design_Number = 1
target_file = f"./data/releases/release_all2(20250620)/release_all2/trojan/design{Test_Design_Number}.v"

trojan_gates = []

parser = NetlistParser()
parser.parse_netlist(target_file)
chain = []

from utils.Tokenizer_functions import extract_trojan_gates

reference_Trojans_file = f"./data/releases/release_all2(20250620)/release_all2/trojan/result{Test_Design_Number}.txt"
actual_trojan_gates = extract_trojan_gates(reference_Trojans_file)
print(f"Actual trojan gates: {sorted(actual_trojan_gates)}")

/home/nadertehrani/Navid/Research/Contest Folders/Contest5/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Actual trojan gates: ['g1070', 'g1154', 'g1204', 'g1243', 'g1341', 'g1360', 'g1362', 'g1396', 'g1413', 'g1494', 'g1530', 'g1599', 'g1646', 'g176', 'g355', 'g549', 'g633', 'g651', 'g661', 'g722', 'g788', 'g823', 'g847', 'g940']


In [4]:
for gate_name in actual_trojan_gates:
    gate = parser.get_gate(gate_name)
    trojan_outputs = []
    trojan_inputs = []
    for output in gate.outputs:
        if output.name in actual_trojan_gates:
            trojan_outputs.append(output.name)
    for input in gate.inputs:
        if input in actual_trojan_gates:
            trojan_inputs.append(input)
    print(f"Gate: {gate.name}, type: {gate.gate_type}, Inputs: {trojan_inputs}, Outputs: {trojan_outputs}")


Gate: g1494, type: dff, Inputs: ['g1070'], Outputs: ['g355']
Gate: g1413, type: dff, Inputs: ['g661'], Outputs: ['g1362']
Gate: g1360, type: dff, Inputs: ['g651'], Outputs: ['g633']
Gate: g722, type: dff, Inputs: ['g176'], Outputs: ['g1530']
Gate: g847, type: not, Inputs: ['g1530'], Outputs: ['g176']
Gate: g1154, type: not, Inputs: ['g633'], Outputs: ['g651', 'g1646']
Gate: g1530, type: nand, Inputs: ['g722'], Outputs: ['g847']
Gate: g633, type: nand, Inputs: ['g1360'], Outputs: ['g1154']
Gate: g1599, type: not, Inputs: ['g355'], Outputs: ['g940', 'g1243']
Gate: g1204, type: not, Inputs: ['g1362'], Outputs: ['g661', 'g788']
Gate: g355, type: nand, Inputs: ['g1494'], Outputs: ['g1599']
Gate: g1362, type: nand, Inputs: ['g1413'], Outputs: ['g1204']
Gate: g549, type: not, Inputs: ['g1396'], Outputs: ['g940', 'g1243']
Gate: g1396, type: not, Inputs: [], Outputs: ['g549']
Gate: g176, type: xnor, Inputs: ['g847', 'g1646'], Outputs: ['g722']
Gate: g651, type: xnor, Inputs: ['g1154', 'g788'], 

In [5]:
g = parser.get_gate("g314")
print(g.input_nets)

['n822']


In [6]:
def find_PIs(gate_name: str) -> set:
    """
    Find all primary inputs (PIs) in the fanin cone of the given gate using bfs.
    """
    gate = parser.get_gate(gate_name)
    pis = set()
    queue = [gate]
    visited = set()
    visited.add(gate.name)
    while queue:
        current_gate = queue.pop(0)
        inputs = current_gate.inputs
        if current_gate.gate_type == 'dff':
            inputs = [current_gate.inputs[3]]
        for input_net in inputs:
            driver_gate = parser.get_gate(input_net)
            if input_net in visited:
                continue
            if driver_gate is None:
                # This net has no driver, it must be a primary input
                # if input_net[0] != '1' and input_net != 'n1':
                if input_net[0] != '1':
                    pis.add(input_net)
            else:
                # Add the driver gate to the queue for further exploration
                queue.append(driver_gate)
                visited.add(driver_gate.name)
    return pis

In [7]:
#print(sorted(find_PIs('g2035')))

In [8]:
gate_fanins = []
for gate_name in actual_trojan_gates:
    pis = find_PIs(gate_name)
    gate_fanins.append((gate_name, len(pis)))
sorted_gates = sorted(gate_fanins, key=lambda x: x[1], reverse=True)
for gate_name, fanin_size in sorted_gates:
    print(f"Gate: {gate_name}, Fanin size: {fanin_size}")

Gate: g1494, Fanin size: 2
Gate: g1413, Fanin size: 2
Gate: g1360, Fanin size: 2
Gate: g722, Fanin size: 2
Gate: g847, Fanin size: 2
Gate: g1154, Fanin size: 2
Gate: g1530, Fanin size: 2
Gate: g633, Fanin size: 2
Gate: g1599, Fanin size: 2
Gate: g1204, Fanin size: 2
Gate: g355, Fanin size: 2
Gate: g1362, Fanin size: 2
Gate: g176, Fanin size: 2
Gate: g651, Fanin size: 2
Gate: g1646, Fanin size: 2
Gate: g661, Fanin size: 2
Gate: g823, Fanin size: 2
Gate: g788, Fanin size: 2
Gate: g1070, Fanin size: 2
Gate: g1341, Fanin size: 2
Gate: g1243, Fanin size: 2
Gate: g940, Fanin size: 2
Gate: g549, Fanin size: 1
Gate: g1396, Fanin size: 1


In [9]:
ans = 0
for gate_name in parser.gates:
    gate = parser.get_gate(gate_name)
    if gate.gate_type != 'dff':
        for i in range(32):
            if f'n30[{i}]' in gate.input_nets:
                ans += 1
                print(gate.name)
                break
print(ans)

# for gate_name in parser.gates:
#     gate = parser.get_gate(gate_name)
#     if gate.gate_type == 'dff':
#         print (gate.name, gate.output_net)

0
